In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 03 — Modelagem Gold
# MAGIC
# MAGIC **Objetivo:** consolidar `silver.clientes` e `silver.historico_credito` em tabelas prontas
# MAGIC para responder as 4 perguntas de negócio do objetivo do MVP.

# COMMAND ----------

CATALOG = "mvp_credito"

# COMMAND ----------

# MAGIC %md
# MAGIC ## `gold.fato_risco_cliente`
# MAGIC Uma linha por cliente, combinando atributos cadastrais com métricas agregadas do bureau.
# MAGIC
# MAGIC **Transformação:** agregação de `silver.historico_credito` por `sk_id_curr` (contagem de
# MAGIC créditos, contagem de créditos ativos, soma de valores e de atrasos) seguida de LEFT JOIN
# MAGIC com `silver.clientes` — LEFT porque nem todo cliente tem histórico no bureau, e isso em si
# MAGIC é informação relevante (pergunta 4).

# COMMAND ----------

from pyspark.sql import functions as F

df_clientes = spark.table(f"{CATALOG}.silver.clientes")
df_bureau = spark.table(f"{CATALOG}.silver.historico_credito")

df_bureau_agg = (
    df_bureau
    .groupBy("sk_id_curr")
    .agg(
        F.count("*").alias("qtd_creditos_bureau"),
        F.sum(F.when(F.col("status_credito") == "Active", 1).otherwise(0)).alias("qtd_creditos_ativos_bureau"),
        F.sum("valor_credito").alias("valor_total_credito_bureau"),
        F.sum("valor_divida_atual").alias("divida_total_bureau"),
        F.sum("valor_em_atraso").alias("valor_total_atraso_bureau"),
        F.max("dias_atraso").alias("max_dias_atraso_bureau"),
    )
)

df_gold_fato = (
    df_clientes
    .join(df_bureau_agg, on="sk_id_curr", how="left")
    .fillna({
        "qtd_creditos_bureau": 0,
        "qtd_creditos_ativos_bureau": 0,
        "valor_total_credito_bureau": 0.0,
        "divida_total_bureau": 0.0,
        "valor_total_atraso_bureau": 0.0,
        "max_dias_atraso_bureau": 0,
    })
    .withColumn("possui_historico_bureau", F.col("qtd_creditos_bureau") > 0)
    .withColumn(
        "faixa_renda",
        F.when(F.col("renda_total") < 100000, "baixa (<100k)")
         .when(F.col("renda_total") < 300000, "média (100k-300k)")
         .otherwise("alta (>300k)")
    )
    .withColumn(
        "faixa_etaria",
        F.when(F.col("idade") < 30, "18-29")
         .when(F.col("idade") < 40, "30-39")
         .when(F.col("idade") < 50, "40-49")
         .when(F.col("idade") < 60, "50-59")
         .otherwise("60+")
    )
)

df_gold_fato.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.fato_risco_cliente")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Tabelas agregadas — uma por pergunta de negócio

# COMMAND ----------

# MAGIC %md
# MAGIC ### `gold.agg_risco_por_segmento` — Pergunta 1
# MAGIC Taxa de inadimplência por faixa de renda, tipo de contrato e escolaridade.

# COMMAND ----------

df_gold_segmento = (
    spark.table(f"{CATALOG}.gold.fato_risco_cliente")
    .groupBy("faixa_renda", "tipo_contrato", "escolaridade")
    .agg(
        F.count("*").alias("qtd_clientes"),
        F.avg("target").alias("taxa_inadimplencia"),
    )
    .filter(F.col("qtd_clientes") >= 30)  # remove segmentos com amostra pequena demais para ser conclusivo
)
df_gold_segmento.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.agg_risco_por_segmento")

# COMMAND ----------

# MAGIC %md
# MAGIC ### `gold.agg_risco_por_bureau` — Pergunta 2
# MAGIC Taxa de inadimplência por quantidade de créditos registrados no bureau.

# COMMAND ----------

df_gold_bureau = (
    spark.table(f"{CATALOG}.gold.fato_risco_cliente")
    .withColumn(
        "faixa_qtd_creditos",
        F.when(F.col("qtd_creditos_bureau") == 0, "0")
         .when(F.col("qtd_creditos_bureau") <= 2, "1-2")
         .when(F.col("qtd_creditos_bureau") <= 5, "3-5")
         .when(F.col("qtd_creditos_bureau") <= 10, "6-10")
         .otherwise("11+")
    )
    .groupBy("faixa_qtd_creditos")
    .agg(
        F.count("*").alias("qtd_clientes"),
        F.avg("target").alias("taxa_inadimplencia"),
    )
)
df_gold_bureau.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.agg_risco_por_bureau")

# COMMAND ----------

# MAGIC %md
# MAGIC ### `gold.agg_risco_por_idade` — Pergunta 3
# MAGIC Taxa de inadimplência por faixa etária e por faixa de tempo de emprego.

# COMMAND ----------

df_gold_idade = (
    spark.table(f"{CATALOG}.gold.fato_risco_cliente")
    .groupBy("faixa_etaria")
    .agg(
        F.count("*").alias("qtd_clientes"),
        F.avg("target").alias("taxa_inadimplencia"),
        F.avg("anos_empregado").alias("media_anos_empregado"),
    )
    .orderBy("faixa_etaria")
)
df_gold_idade.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.agg_risco_por_idade")

# COMMAND ----------

# MAGIC %md
# MAGIC ### `gold.agg_risco_por_divida_externa` — Pergunta 4
# MAGIC Comparação de risco entre clientes com dívida ativa em outras instituições, com histórico
# MAGIC quitado, e sem histórico de bureau algum.

# COMMAND ----------

df_gold_divida = (
    spark.table(f"{CATALOG}.gold.fato_risco_cliente")
    .withColumn(
        "perfil_divida_externa",
        F.when(~F.col("possui_historico_bureau"), "Sem histórico no bureau")
         .when(F.col("divida_total_bureau") > 0, "Com dívida ativa no bureau")
         .otherwise("Histórico quitado, sem dívida ativa")
    )
    .groupBy("perfil_divida_externa")
    .agg(
        F.count("*").alias("qtd_clientes"),
        F.avg("target").alias("taxa_inadimplencia"),
        F.avg("divida_total_bureau").alias("media_divida_bureau"),
    )
)
df_gold_divida.write.mode("overwrite").saveAsTable(f"{CATALOG}.gold.agg_risco_por_divida_externa")

# COMMAND ----------

# MAGIC %md
# MAGIC ### Comentários de catálogo — tabelas Gold

# COMMAND ----------

spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.gold.fato_risco_cliente IS
    'Tabela fato: 1 linha por cliente, combinando atributos cadastrais (silver.clientes) com métricas agregadas do histórico de crédito externo (silver.historico_credito). Base para todas as tabelas agregadas Gold.'
""")
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.gold.agg_risco_por_segmento IS
    'Taxa de inadimplência agregada por faixa de renda, tipo de contrato e escolaridade. Suporta a Pergunta 1 do objetivo. Segmentos com menos de 30 clientes foram excluídos.'
""")
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.gold.agg_risco_por_bureau IS
    'Taxa de inadimplência agregada por faixa de quantidade de créditos registrados no bureau externo. Suporta a Pergunta 2 do objetivo.'
""")
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.gold.agg_risco_por_idade IS
    'Taxa de inadimplência agregada por faixa etária, com média de anos de emprego por faixa. Suporta a Pergunta 3 do objetivo.'
""")
spark.sql(f"""
    COMMENT ON TABLE {CATALOG}.gold.agg_risco_por_divida_externa IS
    'Comparação de taxa de inadimplência entre clientes com dívida ativa no bureau, com histórico quitado e sem histórico externo algum. Suporta a Pergunta 4 do objetivo.'
""")

comentarios_fato = {
    "qtd_creditos_bureau": "Nº total de créditos do cliente reportados por outras instituições. Domínio: inteiro >= 0. Linhagem: COUNT de silver.historico_credito por sk_id_curr.",
    "qtd_creditos_ativos_bureau": 'Nº de créditos com status "Active" no bureau. Domínio: inteiro >= 0. Linhagem: agregado condicional de silver.historico_credito.',
    "valor_total_credito_bureau": "Soma do valor de todos os créditos do cliente no bureau. Domínio: >= 0. Linhagem: SUM de silver.historico_credito.valor_credito.",
    "divida_total_bureau": "Soma do saldo devedor atual de todos os créditos do cliente no bureau. Domínio: >= 0. Linhagem: SUM de silver.historico_credito.valor_divida_atual.",
    "valor_total_atraso_bureau": "Soma dos valores em atraso reportados no bureau. Domínio: >= 0. Linhagem: SUM de silver.historico_credito.valor_em_atraso.",
    "possui_historico_bureau": "Flag indicando se o cliente tem ao menos um registro no bureau externo. Domínio: true/false.",
    "faixa_renda": 'Faixa de renda derivada de renda_total para fins de agregação. Domínio: "baixa (<100k)", "média (100k-300k)", "alta (>300k)".',
    "faixa_etaria": 'Faixa etária derivada de idade para fins de agregação. Domínio: "18-29", "30-39", "40-49", "50-59", "60+".',
}
# escapa aspas simples (') dobrando-as, conforme sintaxe SQL, antes de montar o COMMENT
for coluna, comentario in comentarios_fato.items():
    comentario_sql = comentario.replace("'", "''")
    spark.sql(f"ALTER TABLE {CATALOG}.gold.fato_risco_cliente ALTER COLUMN {coluna} COMMENT '{comentario_sql}'")

# COMMAND ----------

print("gold.fato_risco_cliente:", spark.table(f"{CATALOG}.gold.fato_risco_cliente").count(), "linhas")
display(spark.table(f"{CATALOG}.gold.agg_risco_por_segmento").orderBy(F.desc("taxa_inadimplencia")).limit(10))

gold.fato_risco_cliente: 307511 linhas


faixa_renda,tipo_contrato,escolaridade,qtd_clientes,taxa_inadimplencia
média (100k-300k),Revolving loans,Lower secondary,122,0.13114754098360656
média (100k-300k),Cash loans,Lower secondary,2162,0.1211840888066605
baixa (<100k),Cash loans,Incomplete higher,1122,0.10160427807486631
baixa (<100k),Cash loans,Lower secondary,1362,0.09544787077826726
média (100k-300k),Cash loans,Secondary / secondary special,143425,0.09409796060658882
média (100k-300k),Cash loans,Incomplete higher,7048,0.0876844494892168
baixa (<100k),Cash loans,Secondary / secondary special,47051,0.08614057086990712
alta (>300k),Cash loans,Incomplete higher,862,0.08120649651972157
alta (>300k),Cash loans,Secondary / secondary special,9649,0.08042284174525857
baixa (<100k),Revolving loans,Secondary / secondary special,5774,0.0744717700034638
